#Gold Layer — Business Aggregations in Spark SQL

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS healthcare_catalog.gold")

# 1. Patient count per hospital
spark.sql("""
CREATE OR REPLACE TABLE healthcare_catalog.gold.patient_count_per_hospital AS
SELECT Hospital, COUNT(*) AS patient_count
FROM healthcare_catalog.silver.patients
WHERE scd_is_current = 'Y'
GROUP BY Hospital
ORDER BY patient_count DESC
""")

# 2. Hospital ranking (volume + avg billing + rank)
spark.sql("""
CREATE OR REPLACE TABLE healthcare_catalog.gold.hospital_ranking AS
SELECT
    Hospital,
    COUNT(*) AS patient_count,
    ROUND(AVG(Billing_Amount), 2) AS avg_billing,
    RANK() OVER (ORDER BY COUNT(*) DESC) AS rank_by_volume
FROM healthcare_catalog.silver.patients
WHERE scd_is_current = 'Y'
GROUP BY Hospital
""")

# 3. Contribution by medical condition
spark.sql("""
CREATE OR REPLACE TABLE healthcare_catalog.gold.condition_contribution AS
SELECT
    Medical_Condition,
    COUNT(*) AS patient_count,
    ROUND(AVG(Billing_Amount), 2) AS avg_billing,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_of_total
FROM healthcare_catalog.silver.patients
WHERE scd_is_current = 'Y'
GROUP BY Medical_Condition
ORDER BY patient_count DESC
""")

# 4. Admission type distribution
spark.sql("""
CREATE OR REPLACE TABLE healthcare_catalog.gold.admission_type_distribution AS
SELECT
    Admission_Type,
    COUNT(*) AS patient_count,
    ROUND(AVG(Billing_Amount), 2) AS avg_billing
FROM healthcare_catalog.silver.patients
WHERE scd_is_current = 'Y'
GROUP BY Admission_Type
ORDER BY patient_count DESC
""")

# 5. Insurance provider breakdown
spark.sql("""
CREATE OR REPLACE TABLE healthcare_catalog.gold.insurance_provider_breakdown AS
SELECT
    Insurance_Provider,
    COUNT(*) AS patient_count,
    ROUND(SUM(Billing_Amount), 2) AS total_billing
FROM healthcare_catalog.silver.patients
WHERE scd_is_current = 'Y'
GROUP BY Insurance_Provider
ORDER BY patient_count DESC
""")

print("All Gold tables created.")

In [0]:
display(spark.sql("SELECT * FROM healthcare_catalog.gold.condition_contribution"))
display(spark.sql("SELECT * FROM healthcare_catalog.gold.hospital_ranking ORDER BY rank_by_volume LIMIT 10"))